# 🩻 Día 2 — Segmentación de Radiografías con MedSAM (BATCH)

## 📋 Objetivos del Día 2

✅ Definir la estrategia de prompting para MedSAM (bounding boxes o puntos)  
✅ Aplicar MedSAM a 50-100 radiografías y guardar máscaras  
✅ Evaluar visualmente la calidad de segmentación  
✅ Documentar casos de fallo y causas  

## 🎯 Estrategia

Procesaremos imágenes del dataset CNN desde la carpeta `imagenes_procesadas`

---

## 0️⃣ Instalación de Dependencias

In [1]:
# Instalar segment-anything desde GitHub
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

# Instalar otras dependencias
!pip install -q opencv-python scikit-image tqdm

print("✅ Dependencias instaladas")


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✅ Dependencias instaladas


## 1️⃣ Importar Librerías y Configurar

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import cv2
from pathlib import Path
from tqdm import tqdm
from segment_anything import sam_model_registry, SamPredictor

print("✅ Librerías importadas correctamente")
print(f"   PyTorch: {torch.__version__}")
print(f"   NumPy: {np.__version__}")

✅ Librerías importadas correctamente
   PyTorch: 2.11.0+cu130
   NumPy: 2.4.4


## 2️⃣ Configurar Rutas y Dispositivo

### 📁 Estructura del Dataset

```
CNN_desde_0/
└── imagenes_procesadas/
    ├── train/
    │   ├── si/      ← Imágenes con anatomía correcta
    │   └── no/      ← Imágenes con anatomía incorrecta
    └── test/
        ├── si/
        └── no/
```

### 🎯 Opciones de Procesamiento

Puedes elegir procesar:
- `train/si` - Imágenes de entrenamiento correctas
- `train/no` - Imágenes de entrenamiento incorrectas
- `test/si` - Imágenes de test correctas
- `test/no` - Imágenes de test incorrectas

In [ ]:
# --- CONFIGURACIÓN DE RUTAS ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Dispositivo seleccionado: {device.upper()}\n")

# Ruta del modelo
checkpoint_path = "../Dia1/medsam_vit_b.pth"  # En la misma carpeta que el notebook

# RUTA CORREGIDA: Sube 1 nivel (../../) para ir a la raíz del proyecto
input_dir = "../../CNN_desde_0/imagenes_procesadas/test/si"
output_dir = "mascaras_segmentadas_test_si"

print("📁 RUTAS CONFIGURADAS:")
print(f"   Entrada: {input_dir}")
print(f"   Salida: {output_dir}\n")

# Crear carpeta de salida
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Verificar que exista el checkpoint
if not os.path.exists(checkpoint_path):
    print(f"❌ Checkpoint no encontrado: {checkpoint_path}")
    print(f"   Descárgalo desde: https://drive.google.com/file/d/1UAmWL88roYR7wKlnApw5Bcuzf2iQgk6_/view?usp=sharing")
else:
    print(f"✅ Checkpoint encontrado")

# Verificar que exista la carpeta de entrada
if not os.path.exists(input_dir):
    print(f"\n❌ Carpeta de entrada no encontrada: {input_dir}")
    print(f"   Verifica la ruta y la estructura del proyecto")
    print(f"\n   Directorio actual: {os.getcwd()}")
else:
    # Contar imágenes
    image_extensions = ['.png', '.jpg', '.jpeg', '.bmp', '.tiff']
    image_files = []
    for ext in image_extensions:
        image_files.extend([f for f in os.listdir(input_dir) if f.lower().endswith(ext)])
    
    print(f"\n✅ Carpeta de entrada encontrada")
    print(f"   📸 Imágenes encontradas: {len(image_files)}")
    
    if len(image_files) > 0:
        print(f"   Primeras 5 archivos:")
        for f in image_files[:5]:
            print(f"      - {f}")

🖥️  Dispositivo seleccionado: CPU

📁 RUTAS CONFIGURADAS:
   Entrada: ../../CNN_desde_0/imagenes_procesadas/test/si
   Salida: mascaras_segmentadas_test_si

❌ Checkpoint no encontrado: medsam_vit_b.pth
   Descárgalo desde: https://drive.google.com/file/d/1UAmWL88roYR7wKlnApw5Bcuzf2iQgk6_/view?usp=sharing

✅ Carpeta de entrada encontrada
   📸 Imágenes encontradas: 8
   Primeras 5 archivos:
      - img_1776863909985.png
      - img_1777026517084.png
      - img_1776863889130.png
      - img_1777027170211.png
      - img_1777028264615.png


## 3️⃣ Cargar Modelo MedSAM

### Configuración
- **Modelo**: ViT-B (Vision Transformer Base)
- **Entrada**: Imagen codificada
- **Salida**: Máscara de segmentación + Score de confianza

In [4]:
print("⏳ Cargando modelo MedSAM...\n")

try:
    # Crear instancia
    medsam_model = sam_model_registry["vit_b"]()
    print("   ✓ Modelo instanciado")
    
    # Cargar pesos
    with open(checkpoint_path, "rb") as f:
        state_dict = torch.load(f, map_location="cpu")
    medsam_model.load_state_dict(state_dict)
    print("   ✓ Pesos cargados")
    
    # Mover a dispositivo
    medsam_model.to(device)
    medsam_model.eval()
    print("   ✓ Modelo en dispositivo")
    
    print(f"\n✅ Modelo cargado exitosamente en {device.upper()}")
    
except FileNotFoundError:
    print(f"❌ Error: No se encontró {checkpoint_path}")
    raise
except Exception as e:
    print(f"❌ Error al cargar: {e}")
    raise

⏳ Cargando modelo MedSAM...

   ✓ Modelo instanciado
❌ Error: No se encontró medsam_vit_b.pth


FileNotFoundError: [Errno 2] No such file or directory: 'medsam_vit_b.pth'

## 4️⃣ Definir Estrategia de Prompting

### 📌 Opciones de Input

| Método | Descripción | Ventaja |
|--------|-------------|----------|
| **Bounding Box** | Caja que delimita la estructura | Rápido, automático |
| **Puntos** | Clicks dentro de la anatomía | Más preciso, interactivo |
| **Ambos** | Combinación de box + puntos | Óptimo, pero más lento |

### 🎯 Estrategia Elegida: **Bounding Box Automático**

Detectaremos automáticamente la región más importante de cada radiografía.

In [ ]:
def obtener_bounding_box_automatico(imagen_np, margen=30):
    """
    Detecta automáticamente un bounding box basado en contornos.
    
    Args:
        imagen_np: Array NumPy de la imagen (H×W×3)
        margen: Píxeles adicionales alrededor del contorno
    
    Returns:
        array: [x_min, y_min, x_max, y_max]
    """
    # Convertir a escala de grises
    if len(imagen_np.shape) == 3:
        gray = cv2.cvtColor(imagen_np, cv2.COLOR_RGB2GRAY)
    else:
        gray = imagen_np
    
    # Binarizar
    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    
    # Encontrar contornos
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        # Si no hay contornos, usar toda la imagen
        h, w = gray.shape
        return np.array([0, 0, w, h])
    
    # Encontrar el contorno más grande
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    
    # Aplicar margen
    x_min = max(0, x - margen)
    y_min = max(0, y - margen)
    x_max = min(imagen_np.shape[1], x + w + margen)
    y_max = min(imagen_np.shape[0], y + h + margen)
    
    return np.array([x_min, y_min, x_max, y_max])

print("✅ Función de detección de bounding box definida")

## 5️⃣ Procesar Dataset en Batch

### 🔄 Pipeline

1. **Cargar imagen** → Array NumPy
2. **Detectar box** → Bounding box automático
3. **Segmentar** → MedSAM genera máscara
4. **Guardar** → Máscara en PNG
5. **Registrar** → Score y metadata

### 📊 Salida

- Máscara PNG para cada imagen
- CSV con scores y metadata
- Reporte de errores

In [ ]:
def segmentar_dataset(input_dir, output_dir, medsam_model, predictor, device, max_images=None):
    """
    Procesa un dataset completo de radiografías.
    
    Args:
        input_dir: Carpeta con imágenes
        output_dir: Carpeta para guardar máscaras
        medsam_model: Modelo cargado
        predictor: SamPredictor
        device: 'cuda' o 'cpu'
        max_images: Límite de imágenes a procesar (None = todas)
    
    Returns:
        dict: Estadísticas del procesamiento
    """
    
    # Obtener lista de imágenes
    image_extensions = ['.png', '.jpg', '.jpeg', '.bmp', '.tiff']
    image_files = []
    
    for ext in image_extensions:
        image_files.extend(Path(input_dir).glob(f'*{ext}'))
        image_files.extend(Path(input_dir).glob(f'*{ext.upper()}'))
    
    image_files = sorted(list(set(image_files)))  # Eliminar duplicados y ordenar
    
    if max_images:
        image_files = image_files[:max_images]
    
    print(f"📸 Encontradas {len(image_files)} imágenes")
    print(f"📁 Procesando desde: {input_dir}")
    print(f"💾 Guardando en: {output_dir}\n")
    
    # Inicializar estadísticas
    stats = {
        'total': len(image_files),
        'exitosas': 0,
        'errores': 0,
        'scores': [],
        'archivos_procesados': [],
        'errores_detallados': []
    }
    
    # Procesar cada imagen
    for idx, image_path in enumerate(tqdm(image_files, desc="Procesando radiografías")):
        try:
            # Cargar imagen
            image_pil = Image.open(image_path).convert('RGB')
            image_np = np.array(image_pil)
            
            # Configurar predictor
            predictor.set_image(image_np)
            
            # Detectar bounding box automático
            box = obtener_bounding_box_automatico(image_np)
            
            # Segmentar
            with torch.no_grad():
                masks, scores, _ = predictor.predict(
                    point_coords=None,
                    point_labels=None,
                    box=box[None, :],
                    multimask_output=False
                )
            
            score = scores[0]
            mask = masks[0]
            
            # Guardar máscara
            output_name = image_path.stem + "_mask.png"
            output_path = Path(output_dir) / output_name
            
            # Convertir máscara a uint8 (0-255)
            mask_uint8 = (mask * 255).astype(np.uint8)
            Image.fromarray(mask_uint8).save(output_path)
            
            # Registrar éxito
            stats['exitosas'] += 1
            stats['scores'].append(score)
            stats['archivos_procesados'].append({
                'imagen': image_path.name,
                'mascara': output_name,
                'score': float(score),
                'box': box.tolist()
            })
            
        except Exception as e:
            stats['errores'] += 1
            stats['errores_detallados'].append({
                'imagen': image_path.name if image_path else 'desconocida',
                'error': str(e)
            })
    
    return stats

print("✅ Función de procesamiento batch definida")

## 6️⃣ Ejecutar Segmentación en Batch

### ⚠️ Importante

- **Carpeta de entrada**: Debe contener las radiografías
- **Tiempo estimado**: ~30-60 segundos por imagen (CPU), ~5-10 segundos (GPU)
- **Ajusta `max_images`** si quieres procesar un subconjunto
  - `max_images=10` → Procesa las primeras 10 imágenes (para pruebas rápidas)
  - `max_images=50` → Procesa 50 imágenes
  - `max_images=None` → Procesa TODAS las imágenes

In [ ]:
# Crear predictor
predictor = SamPredictor(medsam_model)

# Ejecutar procesamiento
# AJUSTA max_images según necesites:
# - 10 para una prueba rápida
# - 50-100 para el dataset completo
# - None para procesar TODAS

stats = segmentar_dataset(
    input_dir=input_dir,
    output_dir=output_dir,
    medsam_model=medsam_model,
    predictor=predictor,
    device=device,
    max_images=50  # Cambia aquí
)

print("\n" + "="*60)
print("📊 RESUMEN DEL PROCESAMIENTO")
print("="*60)
print(f"Total procesadas: {stats['total']}")
print(f"Exitosas: {stats['exitosas']} ✅")
print(f"Errores: {stats['errores']} ❌")
print(f"Tasa de éxito: {(stats['exitosas']/stats['total']*100):.1f}%")

if stats['scores']:
    scores_array = np.array(stats['scores'])
    print(f"\n📈 Estadísticas de Scores:")
    print(f"  Media: {scores_array.mean():.4f}")
    print(f"  Mín: {scores_array.min():.4f}")
    print(f"  Máx: {scores_array.max():.4f}")
    print(f"  Desv. Est: {scores_array.std():.4f}")
    
    # Imágenes con baja confianza
    low_score_count = (scores_array < 0.6).sum()
    print(f"\n⚠️  Imágenes con score < 0.6: {low_score_count} ({low_score_count/len(scores_array)*100:.1f}%)")

if stats['errores_detallados']:
    print(f"\n❌ Primeros 5 errores:")
    for error in stats['errores_detallados'][:5]:
        print(f"  - {error['imagen']}: {error['error']}")

## 7️⃣ Guardar Reporte en CSV

### 📋 Contenido del Reporte

- Nombre de imagen
- Nombre de máscara generada
- Score de confianza
- Bounding box utilizado

In [ ]:
import csv
from datetime import datetime

# Crear nombre del reporte con timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
reporte_path = Path(output_dir) / f"reporte_segmentacion_{timestamp}.csv"

# Escribir reporte
with open(reporte_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['imagen', 'mascara', 'score', 'box', 'estado'])
    writer.writeheader()
    
    for archivo in stats['archivos_procesados']:
        estado = "✅ Válido" if archivo['score'] > 0.6 else "⚠️  Revisar"
        writer.writerow({
            'imagen': archivo['imagen'],
            'mascara': archivo['mascara'],
            'score': f"{archivo['score']:.4f}",
            'box': str(archivo['box']),
            'estado': estado
        })

print(f"✅ Reporte guardado en: {reporte_path}")
print(f"📊 Total de registros: {len(stats['archivos_procesados'])}")

## 8️⃣ Visualizar Resultados

### 🎨 Mostrar Ejemplos de Segmentación

Comparamos imagen original vs máscara generada para verificar calidad.

In [ ]:
# Seleccionar imágenes para visualizar (primeras 5)
num_visualizar = min(5, len(stats['archivos_procesados']))

fig, axes = plt.subplots(num_visualizar, 2, figsize=(12, 4*num_visualizar))

if num_visualizar == 1:
    axes = axes.reshape(1, -1)

for idx in range(num_visualizar):
    archivo_info = stats['archivos_procesados'][idx]
    imagen_path = Path(input_dir) / archivo_info['imagen']
    mascara_path = Path(output_dir) / archivo_info['mascara']
    score = archivo_info['score']
    
    try:
        # Cargar imagen original
        imagen = Image.open(imagen_path)
        mascara = Image.open(mascara_path)
        
        # Mostrar imagen
        axes[idx, 0].imshow(imagen)
        axes[idx, 0].set_title(f"Original: {archivo_info['imagen'][:30]}...")
        axes[idx, 0].axis('off')
        
        # Mostrar máscara
        axes[idx, 1].imshow(mascara, cmap='gray')
        color = 'green' if score > 0.6 else 'red'
        axes[idx, 1].set_title(f"Máscara (Score: {score:.4f})", color=color, fontweight='bold')
        axes[idx, 1].axis('off')
        
    except Exception as e:
        print(f"Error visualizando {archivo_info['imagen']}: {e}")

plt.tight_layout()
visualization_path = Path(output_dir) / f"visualizacion_resultados_{timestamp}.png"
plt.savefig(visualization_path, dpi=100, bbox_inches='tight')
plt.show()

print(f"✅ Visualización guardada en: {visualization_path}")

## 9️⃣ Evaluar Calidad de Segmentación

### 🔍 Análisis de Calidad

Preguntas a responder:
- ¿Identifica correctamente mano, pie o tórax?
- ¿Los contornos son precisos?
- ¿Hay sobre/subsegmentación?
- ¿Qué casos fallan?

In [ ]:
# Análisis de scores por rango
scores = np.array(stats['scores'])

print("📊 DISTRIBUCIÓN DE SCORES\n")
print(f"Score > 0.8 (Excelente):     {(scores > 0.8).sum():3d} ({(scores > 0.8).sum()/len(scores)*100:5.1f}%)")
print(f"0.6 < Score ≤ 0.8 (Bueno):   {((scores > 0.6) & (scores <= 0.8)).sum():3d} ({((scores > 0.6) & (scores <= 0.8)).sum()/len(scores)*100:5.1f}%)")
print(f"0.4 < Score ≤ 0.6 (Aceptable): {((scores > 0.4) & (scores <= 0.6)).sum():3d} ({((scores > 0.4) & (scores <= 0.6)).sum()/len(scores)*100:5.1f}%)")
print(f"Score ≤ 0.4 (Pobre):        {(scores <= 0.4).sum():3d} ({(scores <= 0.4).sum()/len(scores)*100:5.1f}%)")

# Visualizar distribución
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
ax1.hist(scores, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax1.axvline(scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {scores.mean():.4f}')
ax1.axvline(0.6, color='orange', linestyle='--', linewidth=2, label='Umbral: 0.6')
ax1.set_xlabel('Score de Confianza', fontsize=12)
ax1.set_ylabel('Número de Imágenes', fontsize=12)
ax1.set_title('Distribución de Scores de Segmentación', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Gráfico de pastel
labels = ['Excelente\n(>0.8)', 'Bueno\n(0.6-0.8)', 'Aceptable\n(0.4-0.6)', 'Pobre\n(≤0.4)']
sizes = [
    (scores > 0.8).sum(),
    ((scores > 0.6) & (scores <= 0.8)).sum(),
    ((scores > 0.4) & (scores <= 0.6)).sum(),
    (scores <= 0.4).sum()
]
colors = ['green', 'yellow', 'orange', 'red']
ax2.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title('Calidad de Segmentación', fontsize=14, fontweight='bold')

plt.tight_layout()
stats_path = Path(output_dir) / f"distribucion_scores_{timestamp}.png"
plt.savefig(stats_path, dpi=100, bbox_inches='tight')
plt.show()

print(f"\n✅ Gráficos guardados en: {stats_path}")

## 🔟 Documentación de Casos de Fallo

### 📝 Registro de Problemas

Analizar y documentar:
- Imágenes con score bajo
- Patrones de fallo
- Posibles causas

In [ ]:
# Encontrar imágenes con bajo score
low_score_threshold = 0.6
low_score_images = [f for f in stats['archivos_procesados'] if f['score'] < low_score_threshold]

print(f"\n⚠️  IMÁGENES CON BAJO SCORE (< {low_score_threshold})\n")
print(f"Total: {len(low_score_images)} de {len(stats['archivos_procesados'])}\n")

if low_score_images:
    for idx, img_info in enumerate(low_score_images[:10], 1):
        print(f"{idx}. {img_info['imagen']}")
        print(f"   Score: {img_info['score']:.4f}")
        print()

# Crear archivo de documentación
doc_path = Path(output_dir) / f"casos_fallo_{timestamp}.txt"
with open(doc_path, 'w') as f:
    f.write("DOCUMENTACIÓN DE CASOS DE FALLO\n")
    f.write("="*60 + "\n\n")
    f.write(f"Threshold: Score < {low_score_threshold}\n")
    f.write(f"Total de casos: {len(low_score_images)}\n\n")
    
    for img_info in low_score_images:
        f.write(f"Imagen: {img_info['imagen']}\n")
        f.write(f"Score: {img_info['score']:.4f}\n")
        f.write(f"\nPosibles causas a investigar:\n")
        f.write(f"  - Iluminación deficiente\n")
        f.write(f"  - Movimiento o artefactos\n")
        f.write(f"  - Anatomía atípica\n")
        f.write(f"  - Bounding box inadecuado\n")
        f.write(f"  - Contraste bajo\n")
        f.write("\n" + "-"*60 + "\n\n")

print(f"✅ Documentación guardada en: {doc_path}")

## 📌 Resumen Final

### ✅ Completado

1. ✅ Estrategia de prompting: Bounding box automático
2. ✅ Procesamiento batch: Radiografías procesadas
3. ✅ Guardado de máscaras: PNG en carpeta output
4. ✅ Evaluación de calidad: Análisis de scores
5. ✅ Documentación: CSV + casos de fallo

### 📂 Archivos Generados

```
mascaras_segmentadas_train_si/
├── imagen_001_mask.png
├── imagen_002_mask.png
├── ...
├── reporte_segmentacion_YYYYMMDD_HHMMSS.csv
├── visualizacion_resultados_YYYYMMDD_HHMMSS.png
├── distribucion_scores_YYYYMMDD_HHMMSS.png
└── casos_fallo_YYYYMMDD_HHMMSS.txt
```

In [ ]:
print("\n" + "="*60)
print("🎉 DÍA 2 — SEGMENTACIÓN COMPLETADA")
print("="*60)
print(f"\n📁 Carpeta de salida: {output_dir}")
print(f"\n📊 Archivos generados:")
print(f"   ✅ {stats['exitosas']} máscaras PNG")
print(f"   ✅ 1 reporte CSV")
print(f"   ✅ 2 gráficos de visualización")
print(f"   ✅ 1 documento de casos de fallo")
print(f"\n✅ Segmentación completada correctamente")
print(f"\n💡 Próximos pasos:")
print(f"   1. Revisar casos con bajo score")
print(f"   2. Ajustar parámetros (margen, umbral)")
print(f"   3. Procesar otros subconjuntos (train/no, test/si, test/no)")
print(f"   4. Evaluar comparativa de calidad entre categorías")